# Final findings: UrbanScape SEO dataset and leakage-safe modeling

This notebook summarizes the honest ML story for the real UrbanScape dataset. The key point is simple: the dataset determines the valid prediction task. We do not force a synthetic decline-label setup onto the real data.

Instead, the project is built around high-performing SEO records, time-aware validation, real feature engineering, and a clear comparison between leakage-safe and leakage-prone feature sets.

## 1) Audit summary

The dataset has a real SEO schema with 9,996 rows and 44 columns. It contains no missing values and no duplicate rows. The date range is valid and the data is temporally structured, but the rows do not form a repeated-entity panel over time. That means the dataset is not well suited to a clean decline prediction task based on repeated content performance over time.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path('..') / 'data' / 'raw' / 'UrbanScape_Apparel_SEO_Performance_Final_Dataset.xlsx'
df = pd.read_excel(data_path)
df['Date'] = pd.to_datetime(df['Date'])

print('rows:', len(df))
print('columns:', len(df.columns))
print('missing_values:', int(df.isna().sum().sum()))
print('duplicate_rows:', int(df.duplicated().sum()))
print('date_min:', df['Date'].min().date())
print('date_max:', df['Date'].max().date())
print('unique_dates:', df['Date'].nunique())

## 2) Valid target definition

Because the data is cross-sectional rather than repeated-entity time series, a realistic target is a binary high-performance flag defined by the observed traffic distribution. We can set the label at the 75th percentile of organic traffic and ask: which records are in the top quartile of performance?

In [ ]:
threshold = df['Organic_Traffic'].quantile(0.75)
df['high_performance_flag'] = (df['Organic_Traffic'] >= threshold).astype(int)
print('traffic_threshold:', round(threshold, 2))
print('positive_rate:', round(df['high_performance_flag'].mean(), 4))

## 3) Baseline comparison

The project compares a leakage-safe set of features against a leakage-prone set designed to exaggerate performance by including variables that either directly encode or closely mirror the outcome window. The purpose is educational: it shows how performance can be inflated when the feature set is not prediction-time valid.

In [ ]:
from src.models.compare_real_baselines import compare_safe_vs_leaky_models

comparison = compare_safe_vs_leaky_models(df)
comparison.sort_values(['feature_set', 'model'])

## 4) Interpretation

The leakage-safe model is the honest estimate of predictive power. The leakage-prone model may look stronger because it uses information that is either outcome-adjacent or conceptually impossible to know at prediction time. That is the teaching point of the project: a high metric alone is not evidence of a valid ML use case.

This project is therefore not about building a production-ready ranking engine. It is about demonstrating disciplined machine learning practice: data-first target selection, explicit leakage checks, time-aware validation, and careful communication of what the model can and cannot do.

## 5) Final conclusion

The real UrbanScape dataset supports a portfolio-quality SEO analysis workflow, including traffic regression, high-performance classification, segmentation analysis, and leakage-risk education. It does not support a clean repeated-entity decline detection task without inventing a pseudo-entity structure.

The honest conclusion is therefore: use the dataset to predict performance in a business-relevant way, but remain explicit about the limits of the data and the validity of the target.